### Implementação dos Modelos de Aprendizado de Máquina (Random Forest, XGBoost, KNN e SVR)

#### Depois do baseline com Regressão Linear Múltipla (05), a ideia aqui é testar modelos mais flexíveis e ver se eles conseguem bater a régua que definimos: **MAE de 5,4 dias e R² de 0,41** no teste.

Para a comparação ser justa, todos os modelos usam **exatamente a mesma base, as mesmas variáveis e a mesma divisão de treino e teste** do baseline (treino até março/2018, teste de abril a agosto/2018). A única coisa que muda é o algoritmo.

In [2]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score

def root_mean_squared_error(y_real, y_prev):
    return np.sqrt(mean_squared_error(y_real, y_prev))
from xgboost import XGBRegressor  # pip install xgboost

-----------------------------------------------------------------------------------------------------

#### I. Recriando o mesmo X e y do baseline

Carrego o `df_modelo.csv` que o 05 salvou (já com uma linha por pedido) e repito os mesmos passos de lá:
* mesmo corte de data;
* `log1p` nas variáveis com cauda longa;
* estados virando dummies;
* sem o `percentual_frete` (que tiramos por causa do VIF);
* alvo do treino limitado entre os percentis 1% e 99%, com o teste intacto.

Deixei tudo numa função só, para não ter risco de algum modelo receber os dados de um jeito diferente.

In [3]:
pedidos = pd.read_csv("../data/processed/df_modelo.csv")
pedidos["order_purchase_timestamp"] = pd.to_datetime(pedidos["order_purchase_timestamp"])

alvo = "dias_atrasados"
numericas = [
    "janela_prometida", "distancia_km", "cross_state", "taxa_atraso_vendedor",
    "volume_historico_vendedor", "preco_total", "frete_total", "peso_total_g",
    "volume_total_cm3", "quantidade_itens_pedido", "vendedores_distintos",
    "payment_installments", "dia_semana_compra", "epoca_alta_demanda",
]
dummies = [c for c in pedidos.columns
           if c.startswith("product_category_name_english_") or c.startswith("payment_type_")]
estados = ["customer_state", "seller_state"]
assimetricas = ["preco_total", "frete_total", "peso_total_g", "volume_total_cm3",
                "distancia_km", "volume_historico_vendedor"]

corte = "2018-04-01"
treino = pedidos[pedidos["order_purchase_timestamp"] < corte].copy()
teste = pedidos[pedidos["order_purchase_timestamp"] >= corte].copy()

def monta_X(base, medias):
    X = base[numericas + dummies + estados].copy()
    for col, media in medias.items():
        X[col] = X[col].fillna(media)
    for col in assimetricas:
        X[col] = np.log1p(X[col])
    return X

medias_treino = treino[["distancia_km", "taxa_atraso_vendedor"]].mean().to_dict()
X_treino = pd.get_dummies(monta_X(treino, medias_treino), columns=estados, drop_first=True, dtype=float)
X_teste = pd.get_dummies(monta_X(teste, medias_treino), columns=estados, drop_first=True, dtype=float)
X_teste = X_teste.reindex(columns=X_treino.columns, fill_value=0)
X_treino, X_teste = X_treino.astype(float), X_teste.astype(float)

limite_inf, limite_sup = treino[alvo].quantile([0.01, 0.99])
y_treino = treino[alvo].clip(limite_inf, limite_sup)
y_teste = teste[alvo]

print(f"Treino: {X_treino.shape} | Teste: {X_teste.shape}")

Treino: (64303, 135) | Teste: (32150, 135)


KNN e SVR trabalham com distâncias, então precisam das variáveis na mesma escala (senão o preço, que vai até milhares, "manda" sozinho). Random Forest e XGBoost são baseados em árvores e não ligam para escala, então uso os dados sem padronizar neles. O scaler é ajustado só no treino.

In [4]:
scaler = StandardScaler()
X_treino_esc = scaler.fit_transform(X_treino)
X_teste_esc = scaler.transform(X_teste)

-----------------------------------------------------------------------------------------------------

#### II. Uma função para treinar e avaliar todo mundo do mesmo jeito

Além de MAE, RMSE e R² (as mesmas do baseline), guardo o tempo de treino e a ROC-AUC da pergunta "vai atrasar?", usando a previsão como nível de risco. Para começar, reproduzo a regressão linear aqui, só para confirmar que os números batem com o 05.

In [5]:
resultados = {}
previsoes = {}
modelos = {}

def avalia(nome, modelo, X_tr, X_te, y_tr=y_treino):
    inicio = time.time()
    modelo.fit(X_tr, y_tr)
    tempo = time.time() - inicio
    prev = modelo.predict(X_te)
    resultados[nome] = {
        "MAE (dias)": mean_absolute_error(y_teste, prev),
        "RMSE (dias)": root_mean_squared_error(y_teste, prev),
        "R²": r2_score(y_teste, prev),
        "ROC-AUC atraso": roc_auc_score((y_teste > 0).astype(int), prev),
        "Tempo de treino (s)": tempo,
    }
    previsoes[nome] = prev
    modelos[nome] = modelo
    return pd.Series(resultados[nome]).round(3)

avalia("Regressão Linear (baseline)", LinearRegression(), X_treino_esc, X_teste_esc)

MAE (dias)             5.407
RMSE (dias)            7.184
R²                     0.406
ROC-AUC atraso         0.754
Tempo de treino (s)    0.933
dtype: float64

Bateu com o 05 (MAE 5,41 e R² 0,41). Agora sim, os modelos novos.

-----------------------------------------------------------------------------------------------------

#### III. Random Forest

A Random Forest treina várias árvores de decisão, cada uma com uma amostra diferente dos dados e das variáveis, e tira a média das previsões. Ela consegue capturar relações não lineares e interações (por exemplo, "distância grande **e** época de Black Friday") que a regressão linear não pega.

Hiperparâmetros que escolhi (sem tuning ainda, é um experimento preliminar):
* `n_estimators=200`: 200 árvores, um meio-termo entre qualidade e tempo.
* `min_samples_leaf=20`: cada folha precisa de pelo menos 20 pedidos, o que evita que a árvore decore casos isolados.
* `max_features=0.5`: cada divisão olha metade das variáveis, para as árvores ficarem mais diferentes entre si.

In [6]:
rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=20, max_features=0.5, n_jobs=-1, random_state=42)
avalia("Random Forest", rf, X_treino, X_teste)

MAE (dias)              5.692
RMSE (dias)             7.493
R²                      0.354
ROC-AUC atraso          0.738
Tempo de treino (s)    20.978
dtype: float64

-----------------------------------------------------------------------------------------------------

#### IV. KNN

O KNN é o mais intuitivo: para prever o atraso de um pedido, ele procura os `k` pedidos mais parecidos no treino e tira a média do atraso deles.

O ponto chato é escolher o `k`. Não quis escolher olhando o teste, porque isso seria "colar" na prova. Então separei os últimos 3 meses do treino (jan–mar/2018) como **validação**, testei alguns valores de `k` ali e só depois treinei com o treino inteiro.

In [7]:
datas_treino = treino["order_purchase_timestamp"].values
eh_validacao = datas_treino >= np.datetime64("2018-01-01")

X_sub, X_val = X_treino_esc[~eh_validacao], X_treino_esc[eh_validacao]
y_sub, y_val = y_treino[~eh_validacao], treino[alvo][eh_validacao]

busca_k = {}
for k in [5, 15, 25, 50, 75, 100]:
    knn_teste = KNeighborsRegressor(n_neighbors=k, weights="distance", n_jobs=-1).fit(X_sub, y_sub)
    busca_k[k] = mean_absolute_error(y_val, knn_teste.predict(X_val))

busca_k = pd.Series(busca_k, name="MAE na validação")
melhor_k = busca_k.idxmin()
print(f"Melhor k na validação: {melhor_k}")
busca_k.round(3).to_frame()

Melhor k na validação: 50


,MAE na validação
5,7.480
15,7.212
25,7.185
50,7.148
75,7.149
100,7.150


In [8]:
knn = KNeighborsRegressor(n_neighbors=melhor_k, weights="distance", n_jobs=-1)
avalia(f"KNN (k={melhor_k})", knn, X_treino_esc, X_teste_esc)

MAE (dias)             6.694
RMSE (dias)            9.087
R²                     0.049
ROC-AUC atraso         0.628
Tempo de treino (s)    0.116
dtype: float64

Desconfio que as ~120 colunas de dummies (categoria e estado) atrapalham o KNN. Com tantas dimensões, "estar perto" vira quase só "ser da mesma categoria e do mesmo estado". Como teste rápido, rodo o KNN só com as variáveis numéricas:

In [9]:
idx_num = [X_treino.columns.get_loc(c) for c in numericas]
knn_num = KNeighborsRegressor(n_neighbors=melhor_k, weights="distance", n_jobs=-1)
avalia(f"KNN (k={melhor_k}, só numéricas)", knn_num, X_treino_esc[:, idx_num], X_teste_esc[:, idx_num])

MAE (dias)             5.954
RMSE (dias)            7.911
R²                     0.280
ROC-AUC atraso         0.700
Tempo de treino (s)    0.293
dtype: float64

-----------------------------------------------------------------------------------------------------

#### V. XGBoost

O XGBoost também usa árvores de decisão, mas as árvores são treinadas em sequência, e cada uma tenta corrigir o erro que as anteriores deixaram. Justamente por ser baseado em árvores, usa os dados sem padronização, como a RF.

Hiperparâmetros (sem tuning ainda, como nos outros modelos):
* `n_estimators=500`: 500 árvores em sequência.
* `learning_rate=0.05`: o quanto cada árvore corrige do erro anterior. Valor baixo aprende com mais cuidado, por isso precisa de mais árvores.
* `max_depth=6`: profundidade máxima de cada árvore, para ela não decorar o treino.
* `subsample=0.8` e `colsample_bytree=0.8`: cada árvore vê 80% dos pedidos e 80% das variáveis, o que reduz o overfitting.

In [10]:
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
)

avalia("XGBoost", xgb, X_treino, X_teste)

MAE (dias)             5.889
RMSE (dias)            7.814
R²                     0.297
ROC-AUC atraso         0.739
Tempo de treino (s)    5.151
dtype: float64

-----------------------------------------------------------------------------------------------------

#### VI. SVR

O SVR (Support Vector Regression) tenta achar uma função que fique dentro de uma "margem" de erro ε para a maioria dos pontos, ignorando erros pequenos. Com o kernel RBF, ele também captura relações não lineares.

O problema prático: o tempo de treino do SVR cresce mais ou menos com o **quadrado** do número de linhas, e com 64 mil pedidos ele demoraria muito. Por isso treino com uma **amostra aleatória de 10 mil pedidos** do treino, o que já é bastante para o SVR. O teste continua sendo o teste completo.

* `C=1`: o quanto o modelo é penalizado por erros fora da margem.
* `epsilon=1`: erros de até 1 dia são tolerados.

In [11]:
amostra_svr = np.random.default_rng(42).choice(len(X_treino_esc), 10_000, replace=False)

svr = SVR(kernel="rbf", C=1.0, epsilon=1.0)
avalia("SVR (amostra de 10 mil)", svr, X_treino_esc[amostra_svr], X_teste_esc,
       y_tr=y_treino.iloc[amostra_svr])

MAE (dias)             5.022
RMSE (dias)            7.155
R²                     0.411
ROC-AUC atraso         0.692
Tempo de treino (s)    8.687
dtype: float64

-----------------------------------------------------------------------------------------------------

#### VII. Otimização dos Hiperparâmetros

Vamos utilizar a busca em grade (Grid Search) para o Random Forest, o XGBoost e o SVR. Cada combinação é treinada com os pedidos anteriores a jan/2018 e avaliada pelo MAE em jan–mar/2018, que é a mesma validação temporal usada no k do KNN. Em contrapartida, o teste não entra nessa escolha.

In [12]:
data_compra = treino["order_purchase_timestamp"]

X_parte = X_treino[data_compra < "2018-01-01"]
y_parte = y_treino[data_compra < "2018-01-01"]

X_validacao = X_treino[data_compra >= "2018-01-01"]
y_validacao = y_treino[data_compra >= "2018-01-01"]

print(len(X_parte), len(X_validacao))

43676 20627


Para o Random Forest, testei duas opções para cada hiperparâmetro:
* `min_samples_leaf` (20 ou 100): número mínimo de pedidos em cada folha, aqui o cuidado é com o possível overfitting.
* `max_features` (0,3 ou 0,5): fração das variáveis sorteada em cada divisão.

In [15]:
rf1 = RandomForestRegressor(n_estimators=200, min_samples_leaf=20, max_features=0.3, n_jobs=-1, random_state=42)
rf1.fit(X_parte, y_parte)
print("RF folha=20  feat=0.3:", mean_absolute_error(y_validacao, rf1.predict(X_validacao)))

rf2 = RandomForestRegressor(n_estimators=200, min_samples_leaf=20, max_features=0.5, n_jobs=-1, random_state=42)
rf2.fit(X_parte, y_parte)
print("RF folha=20  feat=0.5:", mean_absolute_error(y_validacao, rf2.predict(X_validacao)))

rf3 = RandomForestRegressor(n_estimators=200, min_samples_leaf=100, max_features=0.3, n_jobs=-1, random_state=42)
rf3.fit(X_parte, y_parte)
print("RF folha=100 feat=0.3:", mean_absolute_error(y_validacao, rf3.predict(X_validacao)))

rf4 = RandomForestRegressor(n_estimators=200, min_samples_leaf=100, max_features=0.5, n_jobs=-1, random_state=42)
rf4.fit(X_parte, y_parte)
print("RF folha=100 feat=0.5:", mean_absolute_error(y_validacao, rf4.predict(X_validacao)))

RF folha=20  feat=0.3: 6.124010839422431
RF folha=20  feat=0.5: 6.130143792650069
RF folha=100 feat=0.3: 6.20871260949716
RF folha=100 feat=0.5: 6.1858787606456715


Já para o XGBoost, testei duas opções para cada hiperparâmetro, com o número de árvores fixo em 300:
* `max_depth` (3 ou 6): profundidade máxima de cada árvore.
* `learning_rate` (0,05 ou 0,1): o quanto cada árvore corrige o erro das anteriores.

In [16]:
xgb1 = XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42)
xgb1.fit(X_parte, y_parte)
print("XGB prof=3 taxa=0.05:", mean_absolute_error(y_validacao, xgb1.predict(X_validacao)))

xgb2 = XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42)
xgb2.fit(X_parte, y_parte)
print("XGB prof=3 taxa=0.1:", mean_absolute_error(y_validacao, xgb2.predict(X_validacao)))

xgb3 = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42)
xgb3.fit(X_parte, y_parte)
print("XGB prof=6 taxa=0.05:", mean_absolute_error(y_validacao, xgb3.predict(X_validacao)))

xgb4 = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42)
xgb4.fit(X_parte, y_parte)
print("XGB prof=6 taxa=0.1:", mean_absolute_error(y_validacao, xgb4.predict(X_validacao)))

XGB prof=3 taxa=0.05: 5.998387091082292
XGB prof=3 taxa=0.1: 5.988557038003944
XGB prof=6 taxa=0.05: 5.99119366963608
XGB prof=6 taxa=0.1: 6.071202856689937


Por fim, o SVR. Como ele foi o modelo com o menor MAE na primeira rodada, não fazia sentido deixá-lo de fora da otimização. Testei três opções para cada hiperparâmetro:
* `C` (0,1, 1 ou 10): o quanto o modelo é penalizado pelos erros que ficam fora da margem. Valores altos fazem o modelo se esforçar mais para acertar todo mundo, com risco de overfitting.
* `epsilon` (0,5, 1 ou 2 dias): a largura da margem, ou seja, o tamanho do erro que o modelo ignora.

Assim como no treino original, cada combinação é treinada com uma amostra de 10 mil pedidos (agora sorteados só da parte anterior a jan/2018), porque o custo do SVR cresce muito com o número de pedidos. A validação usa todos os pedidos de jan–mar/2018, com os dados padronizados.

In [ ]:
mascara_parte = (data_compra < "2018-01-01").values
X_parte_esc = X_treino_esc[mascara_parte]
X_validacao_esc = X_treino_esc[~mascara_parte]

amostra_parte = np.random.default_rng(42).choice(len(X_parte_esc), 10_000, replace=False)
X_parte_svr = X_parte_esc[amostra_parte]
y_parte_svr = y_parte.iloc[amostra_parte]

busca_svr = []
for C in [0.1, 1, 10]:
    for eps in [0.5, 1, 2]:
        svr_teste = SVR(kernel="rbf", C=C, epsilon=eps)
        svr_teste.fit(X_parte_svr, y_parte_svr)
        mae = mean_absolute_error(y_validacao, svr_teste.predict(X_validacao_esc))
        busca_svr.append({"C": C, "epsilon": eps, "MAE na validação": mae})
        print(f"SVR C={C:<4} epsilon={eps}: {mae}")

busca_svr = pd.DataFrame(busca_svr)
melhor_svr = busca_svr.loc[busca_svr["MAE na validação"].idxmin()]
melhor_C, melhor_eps = float(melhor_svr["C"]), float(melhor_svr["epsilon"])
print(f"\nMelhor combinação do SVR: C={melhor_C}, epsilon={melhor_eps}")

Enfim, para os modelos finais, uso a combinação com menor MAE na validação (Random Forest com folha mínima 20 e 30% das variáveis, XGBoost com profundidade 3 e taxa 0,1, e o SVR com a combinação escolhida na célula acima), treino de novo com o treino inteiro e avalio no teste com a mesma função `avalia` dos outros modelos.

No SVR, o treino final usa a mesma amostra de 10 mil pedidos do modelo inicial (seção VI), para que a comparação entre as duas versões mostre só o efeito dos hiperparâmetros.

In [19]:
rf_otimizado = RandomForestRegressor(n_estimators=200, min_samples_leaf=20, max_features=0.3, n_jobs=-1, random_state=42)
avalia("Random Forest (otimizado)", rf_otimizado, X_treino, X_teste)

MAE (dias)              5.766
RMSE (dias)             7.592
R²                      0.337
ROC-AUC atraso          0.729
Tempo de treino (s)    12.338
dtype: float64

In [20]:
xgb_otimizado = XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42)
avalia("XGBoost (otimizado)", xgb_otimizado, X_treino, X_teste)

MAE (dias)             5.682
RMSE (dias)            7.503
R²                     0.352
ROC-AUC atraso         0.748
Tempo de treino (s)    4.316
dtype: float64

In [ ]:
svr_otimizado = SVR(kernel="rbf", C=melhor_C, epsilon=melhor_eps)
avalia("SVR (otimizado)", svr_otimizado, X_treino_esc[amostra_svr], X_teste_esc,
       y_tr=y_treino.iloc[amostra_svr])

Para ver o efeito da otimização em cada modelo, coloco lado a lado a versão inicial e a otimizada, com a Regressão Linear como referência:

In [ ]:
pares = [
    ("Random Forest", "Random Forest", "Random Forest (otimizado)"),
    ("XGBoost", "XGBoost", "XGBoost (otimizado)"),
    ("SVR", "SVR (amostra de 10 mil)", "SVR (otimizado)"),
]
linhas = []
for nome, inicial, otimizado in pares:
    linhas.append({
        "Modelo": nome,
        "MAE inicial": resultados[inicial]["MAE (dias)"],
        "MAE otimizado": resultados[otimizado]["MAE (dias)"],
        "RMSE inicial": resultados[inicial]["RMSE (dias)"],
        "RMSE otimizado": resultados[otimizado]["RMSE (dias)"],
    })
efeito_otimizacao = pd.DataFrame(linhas).set_index("Modelo")

base = resultados["Regressão Linear (baseline)"]
print(f"Referência - Regressão Linear: MAE {base['MAE (dias)']:.3f} | RMSE {base['RMSE (dias)']:.3f}")
efeito_otimizacao.round(3)

A otimização melhorou o XGBoost (MAE de 5,89 para 5,68 e R² de 0,30 para 0,35), mas não o Random Forest (MAE de 5,69 para 5,77). A combinação que foi melhor em jan–mar/2018 não foi necessariamente a melhor no período de teste (abr–ago/2018).

As diferenças entre as combinações na validação foram pequenas, de cerca de 0,1 dia, o que indica que o desempenho desses modelos depende mais dos atributos do que dos hiperparâmetros. Nem o Random Forest nem o XGBoost otimizados superaram a Regressão Linear (MAE de 5,41).

O efeito da otimização no SVR está na tabela acima. Com isso, a otimização fica concluída para os três modelos com hiperparâmetros mais sensíveis (Random Forest, XGBoost e SVR), além do k do KNN, escolhido na seção IV.

-----------------------------------------------------------------------------------------------------

#### VIII. Registro dos resultados

Tabela com as métricas de todos os modelos no conjunto de teste, incluindo os modelos otimizados.

In [21]:
tabela = pd.DataFrame(resultados).T.sort_values("MAE (dias)")
tabela.round(3)

,MAE (dias),RMSE (dias),R²,ROC-AUC atraso,Tempo de treino (s)
SVR (amostra de 10 mil),5.022,7.155,0.411,0.692,8.687
Regressão Linear (baseline),5.407,7.184,0.406,0.754,0.933
XGBoost (otimizado),5.682,7.503,0.352,0.748,4.316
Random Forest,5.692,7.493,0.354,0.738,20.978
Random Forest (otimizado),5.766,7.592,0.337,0.729,12.338
XGBoost,5.889,7.814,0.297,0.739,5.151
"KNN (k=50, só numéricas)",5.954,7.911,0.280,0.700,0.293
KNN (k=50),6.694,9.087,0.049,0.628,0.116


In [22]:
pd.DataFrame(resultados).T.round(4).to_csv("../data/processed/resultados_modelos_ml.csv")
print("Salvo: resultados_modelos_ml.csv")

Salvo: resultados_modelos_ml.csv
